# project_tools

> MCP tools for project selection, discovery, and bookmarking

In [ ]:
#| default_exp tools.project

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import os
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.config import load_bookmarks, save_bookmarks
from nbdev_mcp.utils.paths import (
    expand,
    project_summary,
    is_nbdev_project,
    resolve_selector,
    require_project,
    settings_dict,
    nbdev_settings_path,
    nbdev_generation,
)
from nbdev_mcp.utils.rich import render_result, render_table


## Project Selection

In [ ]:
#| export
def set_project(selector: str) -> Dict[str, Any]:
    """Tool: Select an nbdev project to make it active (by path or alias).
    
    Args:
        selector: Project path, alias (prefixed with @), or bookmark name.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    """
    try:
        p = resolve_selector(selector)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    # Use set_current_project to update both config and paths modules
    from nbdev_mcp.utils.config import set_current_project
    set_current_project(p)
    
    meta = project_summary(p)
    pretty = render_result('Project selected', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def current_project() -> Dict[str, Any]:
    """Tool: Show the currently active project's summary information.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    
    Raises:
        RuntimeError: If no project is currently selected.
    """
    p = require_project()
    meta = project_summary(p)
    pretty = render_result('Current project', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def console_scripts_status(project: Optional[str] = None) -> Dict[str, Any]:
    """Tool: Show console_scripts entry points and where to configure them.
    
    Args:
        project: Project path or alias. Uses current project if not specified.
    
    Returns:
        Result dict with 'entries' list and suggestions for adding scripts.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    settings = settings_dict(p)
    settings_file = nbdev_settings_path(p)
    settings_name = settings_file.name if settings_file is not None else 'settings.ini'

    cs = settings.get('console_scripts', '').strip()
    entries = [entry for entry in cs.split() if entry] if cs else []

    if entries:
        msg = f'console_scripts present in {settings_name}.'
    else:
        if settings_name.endswith('.toml'):
            example = 'console_scripts = ["mycli=mypkg:main"]'
        else:
            example = 'console_scripts = mycli=mypkg:main'
        msg = f'No console_scripts configured. Add e.g. `{example}` in {settings_name}.'

    pretty = render_result('console_scripts', {'entries': entries or 'None', 'settings_file': settings_name}, {})
    return {
        'ok': True,
        'entries': entries,
        'settings_file': settings_name,
        'nbdev_generation': nbdev_generation(p),
        'message': msg,
        'pretty': pretty,
    }


## Project Discovery

In [ ]:
#| export
def find_projects(
    roots: Optional[List[str]] = None,
    max_results: int = 50
) -> Dict[str, Any]:
    """Tool: Scan given directories (or common defaults) for nbdev projects.
    
    Searches in provided roots, NBDEV_PROJECTS env var, and common
    directories like ~/code, ~/projects, ~/repos, ~/src, ~/Dev.
    
    Args:
        roots: Additional directories to scan.
        max_results: Maximum number of projects to return (default 50).
    
    Returns:
        Result dict with 'results' list of project summaries.
    """
    search_dirs: List[Path] = []
    
    if roots:
        search_dirs += [expand(r) for r in roots]
    
    env = os.environ.get('NBDEV_PROJECTS')
    if env:
        for r in env.split(os.pathsep):
            pr = expand(r)
            if pr.exists():
                search_dirs.append(pr)
    
    home = Path.home()
    for sub in ('code', 'projects', 'repos', 'src', 'Dev', 'dev', 'Documents'):
        d = home / sub
        if d.exists():
            search_dirs.append(d)
    
    seen, results = set(), []
    for base in search_dirs:
        if not base.is_dir():
            continue
        for p in base.iterdir():
            if p.is_dir() and p not in seen:
                try:
                    if is_nbdev_project(p):
                        results.append(project_summary(p))
                        seen.add(p)
                except Exception:
                    continue
        if len(results) >= max_results:
            break
    
    rows = [[r.get('lib_name') or '?', r['project'], r['nbs_dir']] for r in results]
    pretty = render_table('Discovered nbdev projects', ['lib_name', 'path', 'nbs_dir'], rows)
    
    return {'ok': True, 'results': results, 'pretty': pretty}

## Bookmarks

In [ ]:
#| export
def bookmark_project(alias: str, path: str) -> Dict[str, Any]:
    """Tool: Bookmark an nbdev project path with a short alias name.
    
    Args:
        alias: Short name to use as bookmark (e.g., 'myproj').
        path: Path to the nbdev project directory.
    
    Returns:
        Result dict with 'alias' and 'path' of the saved bookmark.
    """
    try:
        root = resolve_selector(path)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    aliases = load_bookmarks()
    aliases[alias] = str(root)
    save_bookmarks(aliases)
    
    meta = {'alias': alias, 'path': str(root)}
    pretty = render_result('Bookmarked project', meta)
    return {'ok': True, **meta, 'pretty': pretty}

In [ ]:
#| export
def list_bookmarks() -> Dict[str, Any]:
    """Tool: List all saved project bookmarks (alias -> path).
    
    Returns:
        Result dict with 'aliases' mapping bookmark names to project paths.
    """
    aliases = load_bookmarks()
    rows = [[k, v] for k, v in aliases.items()]
    pretty = render_table('Project bookmarks', ['alias', 'path'], rows)
    return {'ok': True, 'aliases': aliases, 'pretty': pretty}

In [ ]:
#| export
def remove_bookmark(alias: str) -> Dict[str, Any]:
    """Tool: Remove a saved project bookmark by alias.
    
    Args:
        alias: Bookmark name to remove.
    
    Returns:
        Result dict with removed alias and path, or error if not found.
    """
    aliases = load_bookmarks()
    if alias in aliases:
        path = aliases.pop(alias)
        save_bookmarks(aliases)
        meta = {'alias': alias, 'removed': path}
        pretty = render_result('Removed bookmark', meta)
        return {'ok': True, **meta, 'pretty': pretty}
    return {'ok': False, 'error': f'No such alias: {alias}'}

In [ ]:
#| export
def config_status() -> Dict[str, Any]:
    """Tool: Show current nbdev-mcp configuration settings.

    Returns:
        Result dict with 'config' settings, 'bookmarks_path', and 'env_overrides'.
    """
    from nbdev_mcp.utils.config import get_config, BOOKMARKS_PATH

    cfg = get_config()
    config_data = {
        'log_level': cfg.get('log_level', 'INFO'),
        'prompt_dir': cfg.get('prompt_dir', 'prompt_templates'),
        'env_dir_name': cfg.get('env_dir_name', 'Environments'),
        'max_tree_files': cfg.get('max_tree_files', 600),
    }

    # Check for environment variable overrides
    env_overrides = {}
    for key in ['LOG_LEVEL', 'PROMPT_DIR', 'ENV_DIR_NAME', 'MAX_TREE_FILES']:
        env_key = f'NBMCP_{key}'
        if env_key in os.environ:
            env_overrides[env_key] = os.environ[env_key]

    pretty = render_result('Configuration', config_data, {
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides or 'None'
    })

    return {
        'ok': True,
        'config': config_data,
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides,
        'pretty': pretty
    }

In [ ]:
#| export
def prompt_templates_status() -> Dict[str, Any]:
    """Tool: List available prompt templates and their locations.

    Returns:
        Result dict with 'templates' list and 'prompt_dir' setting.
    """
    from nbdev_mcp.utils.config import get_config, EXPECTED_PROMPT_TEMPLATES
    from nbdev_mcp.prompts import get_bundled_template

    cfg = get_config()
    prompt_dir = cfg.get('prompt_dir', 'prompt_templates')

    # List of expected templates
    template_names = EXPECTED_PROMPT_TEMPLATES

    templates = []
    for name in template_names:
        try:
            content = get_bundled_template(name)
            templates.append({
                'name': name,
                'path': f'nbdev_mcp.prompt_templates/{name}',
                'exists': True,
                'size': len(content)
            })
        except FileNotFoundError:
            templates.append({
                'name': name,
                'path': 'not found',
                'exists': False,
                'size': 0
            })

    rows = [[t['name'], t['path'], '✓' if t['exists'] else '✗'] for t in templates]
    pretty = render_table('Prompt Templates', ['name', 'path', 'exists'], rows)

    return {
        'ok': True,
        'templates': templates,
        'prompt_dir': prompt_dir,
        'pretty': pretty
    }

In [ ]:
#| export
def health_check() -> Dict[str, Any]:
    """Tool: Validate MCP connection and report system status.
    
    Returns status of:
    - MCP server connection
    - Current project (if any)
    - Key dependencies
    - Available tools count
    
    Returns
    -------
    Dict[str, Any]
        Result with health status, version info, and diagnostics.
    """
    import sys
    import platform
    
    # Check package versions
    deps = {}
    for pkg in ['nbdev', 'mcp', 'fastcore', 'rich']:
        try:
            from importlib.metadata import version as pkg_version
            deps[pkg] = pkg_version(pkg)
        except Exception:
            deps[pkg] = 'not installed'
    
    # Get nbdev-mcp version
    try:
        from importlib.metadata import version as pkg_version
        nbdev_mcp_version = pkg_version('nbdev-mcp')
    except Exception:
        nbdev_mcp_version = 'unknown'
    
    # Check current project
    from nbdev_mcp.utils.config import CURRENT_PROJECT
    project_status = {
        'has_project': CURRENT_PROJECT is not None,
        'path': str(CURRENT_PROJECT) if CURRENT_PROJECT else None
    }
    
    # Build health report
    health = {
        'status': 'healthy',
        'version': nbdev_mcp_version,
        'python': sys.version.split()[0],
        'platform': platform.system(),
        'dependencies': deps,
        'project': project_status,
    }
    
    # Check for issues
    issues = []
    if deps.get('nbdev') == 'not installed':
        issues.append('nbdev is not installed')
        health['status'] = 'degraded'
    if deps.get('mcp') == 'not installed':
        issues.append('mcp is not installed')
        health['status'] = 'error'
    
    if issues:
        health['issues'] = issues
    
    from nbdev_mcp.utils.rich import render_result
    pretty = render_result('Health Check', health)
    return {'ok': health['status'] != 'error', **health, 'pretty': pretty}

In [ ]:
#| export
def mcp_scaffold_guide(
    server_name: str = "mcp.custom",
    module_prefix: str = "50_mcp",
) -> Dict[str, Any]:
    """Tool: Return a notebook-first scaffold guide for building a new MCP server.

    The guide follows this project's conventions:
    - All scripting logic lives under ``nbs/``
    - Exported APIs are surfaced via nbdev settings
    - Generated ``.py`` modules are treated as build artifacts
    """
    notebooks = [
        f"nbs/{module_prefix}/00__init__.ipynb",
        f"nbs/{module_prefix}/01_server.ipynb",
        f"nbs/{module_prefix}/02_tools.ipynb",
        f"nbs/{module_prefix}/03_prompts.ipynb",
        f"nbs/{module_prefix}/04_cli.ipynb",
    ]

    exported_symbols = [
        "create_server",
        "add_tools",
        "add_prompts",
        "main",
    ]

    checklist = [
        "Create notebooks under nbs/ with one responsibility per notebook.",
        "Set #| default_exp and #| export cells for public API.",
        "Wire resources/tools/prompts in a create_server(...) composition root.",
        "Add console_scripts entry in settings.ini for the new CLI entrypoint.",
        "Run nbdev_export after notebook edits.",
        "Add tests for tool registration and config mutation behavior.",
    ]

    result = {
        "ok": True,
        "server_name": server_name,
        "notebooks": notebooks,
        "exported_symbols": exported_symbols,
        "checklist": checklist,
    }
    pretty = render_result("MCP Scaffold Guide", result)
    return {**result, "pretty": pretty}


## MCP Registration

In [ ]:
#| export
# Tool annotation definitions for project tools
TOOL_ANNOTATIONS = {
    'set_project': ToolAnnotations(
        title="Set Project",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'current_project': ToolAnnotations(
        title="Current Project",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'console_scripts_status': ToolAnnotations(
        title="Console Scripts Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'find_projects': ToolAnnotations(
        title="Find Projects",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=True
    ),
    'bookmark_project': ToolAnnotations(
        title="Bookmark Project",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'list_bookmarks': ToolAnnotations(
        title="List Bookmarks",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'remove_bookmark': ToolAnnotations(
        title="Remove Bookmark",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'config_status': ToolAnnotations(
        title="Config Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'prompt_templates_status': ToolAnnotations(
        title="Prompt Templates Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'health_check': ToolAnnotations(
        title="Health Check",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'server_metrics': ToolAnnotations(
        title="Server Metrics",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'analyze_remote': ToolAnnotations(
        title="Analyze Remote",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=True
    ),
    'mcp_scaffold_guide': ToolAnnotations(
        title="MCP Scaffold Guide",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
}

def add_project_tools(mcp: FastMCP) -> None:
    """Attach project management tools to the MCP server with annotations.

    Args:
        mcp: FastMCP server instance.
    """
    tools = [
        ('set_project', set_project),
        ('current_project', current_project),
        ('console_scripts_status', console_scripts_status),
        ('find_projects', find_projects),
        ('bookmark_project', bookmark_project),
        ('list_bookmarks', list_bookmarks),
        ('remove_bookmark', remove_bookmark),
        ('config_status', config_status),
        ('prompt_templates_status', prompt_templates_status),
        ('health_check', health_check),
        ('server_metrics', server_metrics),
        ('analyze_remote', analyze_remote),
        ('mcp_scaffold_guide', mcp_scaffold_guide),
    ]

    for name, func in tools:
        annotations = TOOL_ANNOTATIONS.get(name)
        mcp.tool(name=name, annotations=annotations)(func)


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
import tempfile
import shutil

def analyze_remote(
    url: str,
    branch: str = "main",
    shallow: bool = True,
    timeout: Optional[int] = None
) -> Dict[str, Any]:
    """Tool: Analyze a remote nbdev project without permanent cloning.
    
    Clones a GitHub repository temporarily, analyzes its structure,
    and returns information about the project. Useful for comparing
    with local projects or understanding external nbdev projects.
    
    Args:
        url: GitHub URL (https://github.com/user/repo) or shorthand (user/repo).
        branch: Branch to analyze (default: main).
        shallow: If True, do a shallow clone for faster analysis.
        timeout: Timeout in seconds for cloning (default from config/environment).
    
    Returns:
        Result dict with project info, structure, and analysis.
    """
    import subprocess
    from nbdev_mcp.utils.config import get_config
    
    cfg = get_config()
    if timeout is None:
        timeout = cfg.timeout_analyze_remote
    
    # Normalize URL
    if not url.startswith('http'):
        url = f'https://github.com/{url}'
    
    # Extract repo name
    repo_name = url.rstrip('/').split('/')[-1]
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]
    
    # Create temp directory
    temp_dir = tempfile.mkdtemp(prefix='nbdev_mcp_')
    clone_path = Path(temp_dir) / repo_name
    
    try:
        # Clone the repo
        cmd = ['git', 'clone']
        if shallow:
            cmd.extend(['--depth', '1'])
        cmd.extend(['-b', branch, url, str(clone_path)])
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            return {
                'ok': False,
                'error': f'Clone failed: {result.stderr}',
                'url': url,
                'branch': branch
            }

        # Check if it's an nbdev project
        if not is_nbdev_project(clone_path):
            return {
                'ok': False,
                'error': 'Not an nbdev project (expected nbs/ plus settings.ini/settings.toml/pyproject.toml [tool.nbdev])',
                'url': url
            }

        settings = settings_dict(clone_path)
        settings_file = nbdev_settings_path(clone_path)
        generation = nbdev_generation(clone_path)

        # Find notebooks
        nbs_path = settings.get('nbs_path', 'nbs')
        nbs_dir = clone_path / nbs_path

        notebooks = []
        if nbs_dir.exists():
            notebooks = sorted([
                str(nb.relative_to(clone_path))
                for nb in nbs_dir.rglob('*.ipynb')
                if not nb.name.startswith('.')
            ])

        # Find lib path
        lib_path = settings.get('lib_path', settings.get('lib_name', ''))
        lib_dir = clone_path / lib_path if lib_path else clone_path

        modules = []
        if lib_dir.exists():
            modules = sorted([
                str(m.relative_to(clone_path))
                for m in lib_dir.rglob('*.py')
                if not m.name.startswith('.')
            ])

        requirements_raw = settings.get('requirements', '')

        # Build analysis
        analysis = {
            'ok': True,
            'url': url,
            'branch': branch,
            'name': settings.get('lib_name', repo_name),
            'version': settings.get('version', 'unknown'),
            'description': settings.get('description', ''),
            'author': settings.get('author', ''),
            'requirements': requirements_raw.split() if requirements_raw else [],
            'nbdev_generation': generation,
            'nbdev_settings_file': settings_file.name if settings_file is not None else None,
            'notebooks': {
                'count': len(notebooks),
                'paths': notebooks[:50],  # Limit to 50
            },
            'modules': {
                'count': len(modules),
                'paths': modules[:50],
            },
            'settings': {k: v for k, v in settings.items() if k not in ['requirements', 'dev_requirements']},
        }
        
        from nbdev_mcp.utils.rich import render_result
        pretty = render_result(f'Remote Analysis: {repo_name}', {
            'name': analysis['name'],
            'version': analysis['version'],
            'nbdev_generation': analysis['nbdev_generation'],
            'settings_file': analysis['nbdev_settings_file'] or 'N/A',
            'notebooks': analysis['notebooks']['count'],
            'modules': analysis['modules']['count'],
            'description': analysis['description'][:100] if analysis['description'] else 'N/A'
        })
        
        return {**analysis, 'pretty': pretty}
        
    except subprocess.TimeoutExpired:
        return {'ok': False, 'error': 'Clone timed out', 'url': url}
    except Exception as e:
        return {'ok': False, 'error': str(e), 'url': url}
    finally:
        # Clean up temp directory
        try:
            shutil.rmtree(temp_dir)
        except Exception:
            pass


In [ ]:
#| export
import time

# Module-level tracking for server metrics
_SERVER_START_TIME: Optional[float] = None;
_REQUEST_COUNT: int = 0;
_TOTAL_LATENCY_MS: float = 0.0;

def _init_metrics():
    """Initialize server metrics on first call."""
    global _SERVER_START_TIME
    if _SERVER_START_TIME is None:
        _SERVER_START_TIME = time.time()

def _record_request(latency_ms: float):
    """Record a request's latency."""
    global _REQUEST_COUNT, _TOTAL_LATENCY_MS
    _REQUEST_COUNT += 1
    _TOTAL_LATENCY_MS += latency_ms

def server_metrics() -> Dict[str, Any]:
    """Tool: Return server performance and health metrics.
    
    Returns comprehensive server status including:
    - Uptime
    - Memory usage
    - Request statistics
    - System information
    
    Returns
    -------
    Dict[str, Any]
        Result with uptime, memory, request stats, and system info.
    """
    import sys
    import platform
    import os
    
    _init_metrics()
    
    # Calculate uptime
    uptime_seconds = time.time() - _SERVER_START_TIME if _SERVER_START_TIME else 0
    uptime_hours = uptime_seconds / 3600
    
    # Memory usage
    try:
        import psutil
        process = psutil.Process(os.getpid())
        memory_mb = process.memory_info().rss / (1024 * 1024)
        memory_percent = process.memory_percent()
        cpu_percent = process.cpu_percent()
    except ImportError:
        # psutil not available
        memory_mb = None
        memory_percent = None
        cpu_percent = None
    
    # Request stats
    avg_latency = _TOTAL_LATENCY_MS / _REQUEST_COUNT if _REQUEST_COUNT > 0 else 0
    
    # Build metrics
    metrics = {
        'ok': True,
        'uptime': {
            'seconds': round(uptime_seconds, 1),
            'hours': round(uptime_hours, 2),
            'started_at': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_SERVER_START_TIME)) if _SERVER_START_TIME else None
        },
        'requests': {
            'count': _REQUEST_COUNT,
            'avg_latency_ms': round(avg_latency, 2),
            'total_latency_ms': round(_TOTAL_LATENCY_MS, 2)
        },
        'memory': {
            'rss_mb': round(memory_mb, 1) if memory_mb else 'N/A (install psutil)',
            'percent': round(memory_percent, 1) if memory_percent else 'N/A',
        },
        'cpu': {
            'percent': round(cpu_percent, 1) if cpu_percent else 'N/A (install psutil)',
        },
        'system': {
            'python': sys.version.split()[0],
            'platform': platform.system(),
            'arch': platform.machine(),
            'pid': os.getpid()
        }
    }
    
    from nbdev_mcp.utils.rich import render_table, render_panel
    
    rows = [
        ['Uptime', f"{metrics['uptime']['hours']} hours"],
        ['Requests', str(metrics['requests']['count'])],
        ['Avg Latency', f"{metrics['requests']['avg_latency_ms']} ms"],
        ['Memory', f"{metrics['memory']['rss_mb']} MB" if memory_mb else 'N/A'],
        ['CPU', f"{metrics['cpu']['percent']}%" if cpu_percent else 'N/A'],
        ['Python', metrics['system']['python']],
        ['Platform', f"{metrics['system']['platform']} {metrics['system']['arch']}"],
    ]
    pretty = render_table('Server Metrics', ['Metric', 'Value'], rows)
    
    return {**metrics, 'pretty': pretty}